# Lab 10 – ML Pipelines

We are going to focus on preparing a data set by cleaning the data, creating new features, which are fields that will serve in training the model later, and then looking at selecting a curated set of features based on how promising they look.

Lab based on book: Data Analysis with Python and PySpark, Jonathan Rioux

## 1. Reading, exploring, and preparing our machine learning data set

We will start with the ingestion and exploration of our machine learning data set. More specifically, we’ll review the content of our data frame, look at incoherences, and prepare our data for feature engineering. For our ML model, I chose a data set of 20,057 dish names that contain 680 columns characterizing the ingredient list, the nutritional content, and the category of the dish. 

Dataset source: https://www.kaggle.com/datasets/hugodarwood/epirecipes

👍 **Our goal here is to predict if this dish is a dessert**

In [1]:
#Checking the installed Java version
!java -version
!pip install pyspark 
# Install Java 17
!sudo apt-get update
!sudo apt-get install -y openjdk-17-jdk-headless

!java -version

openjdk version "17.0.16" 2025-07-15
OpenJDK Runtime Environment (build 17.0.16+8-Ubuntu-0ubuntu124.04.1)
OpenJDK 64-Bit Server VM (build 17.0.16+8-Ubuntu-0ubuntu124.04.1, mixed mode, sharing)
Hit:1 https://packages.cloud.google.com/apt cloud-sdk InRelease
Hit:2 https://cli.github.com/packages stable InRelease                         
Hit:3 https://download.docker.com/linux/ubuntu noble InRelease                 
Hit:4 https://security.ubuntu.com/ubuntu noble-security InRelease              
Hit:5 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble InRelease          
Hit:6 https://cloud.archive.ubuntu.com/ubuntu noble InRelease                  
Hit:7 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:8 https://cloud.archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:9 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:10 https://cloud.archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:11 http://deb.wakemeops.com/wakemeops 

In [2]:
# Set JAVA_HOME to Java 17
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .master("local[*]")\
        .appName("Pipelines") \
        .config("spark.sql.ansi.enabled", "false") \
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/09 18:52:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


#### 1.1 Import and clean

In [3]:
import pyspark.sql.functions as F
import pyspark.sql.types as T

In [4]:
# File location and type
file_location = "./epi_r.csv"
file_type = "csv"

# CSV options
infer_schema = "true"
first_row_is_header = "true"
delimiter = ","

# The applied options are for CSV files. For other file types, these will be ignored.
food = spark.read.format(file_type) \
  .option("inferSchema", infer_schema) \
  .option("header", first_row_is_header) \
  .option("sep", delimiter) \
  .load(file_location)

In [5]:
print(food.count(), len(food.columns))

20057 680


In [6]:
food.printSchema()

root
 |-- title: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- calories: string (nullable = true)
 |-- protein: double (nullable = true)
 |-- fat: double (nullable = true)
 |-- sodium: double (nullable = true)
 |-- #cakeweek: double (nullable = true)
 |-- #wasteless: double (nullable = true)
 |-- 22-minute meals: double (nullable = true)
 |-- 3-ingredient recipes: double (nullable = true)
 |-- 30 days of groceries: double (nullable = true)
 |-- advance prep required: double (nullable = true)
 |-- alabama: double (nullable = true)
 |-- alaska: double (nullable = true)
 |-- alcoholic: double (nullable = true)
 |-- almond: double (nullable = true)
 |-- amaretto: double (nullable = true)
 |-- anchovy: double (nullable = true)
 |-- anise: double (nullable = true)
 |-- anniversary: double (nullable = true)
 |-- anthony bourdain: double (nullable = true)
 |-- aperitif: double (nullable = true)
 |-- appetizer: double (nullable = true)
 |-- apple: double (nullable = true)


Some of the columns contains undesirable characters, such as a # (`#cakeweek`), or a space (`30 days of groceries`), or some invalid characters (`bon app��tit`)!

**Having a consistent column naming scheme will make subsequent code easier to write, read, and maintain in the long run.**

We will remove anything that isn’t a letter or a number, standardize the spaces and other separators to use the underscore (_) character, andreplace the ampersand (&) with its English equivalent and.

To apply our function `sanitize_column_name`, we used `toDF()`: when used to rename the colum ns of a data frame, takes as parameters N strings, where N is the number of columns in our data frame. Since we can access the columns of our data frame via `food.columns`, a quick list comprehension takes care of renaming everything. We also unpack my list into distinct attributes using the star operator.

In [7]:
import re

def sanitize_column_name(name: str) -> str:
    """
    Drops unwanted characters from the column name, making it more concise and robust.

    1. Standardizes separators (space, dash, slash) to an underscore.
    2. Replaces ampersand (&) with 'and'.
    3. Retains only alphanumeric characters and underscores using regex.
    
    Args:
        name: The original column name string.
    
    Returns:
        The sanitized column name string.
    """
    
    # Step 1: Standardize common separators and perform the '&' replacement
    # Chaining 'replace' is clear for these specific transformations.
    temp_name = name.replace(" ", "_").replace("-", "_").replace("/", "_").replace("&", "and")

    # Step 2: Use regex to filter out all non-word characters.
    # The pattern '[^\w]' matches any character that is NOT a word character (a-z, A-Z, 0-9, or _).
    # This replaces the entire list comprehension filter with a single, highly optimized call.
    return re.sub(r'[^\w]', '', temp_name)

In [8]:
sanitized_column_names = [sanitize_column_name(name) for name in food.columns]

# Apply the new list of names to the DataFrame using the .toDF() method.
# The asterisk (*) unpacks the list of names into positional arguments for toDF().
food = food.toDF(*sanitized_column_names)


In [9]:
food.printSchema()

root
 |-- title: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- calories: string (nullable = true)
 |-- protein: double (nullable = true)
 |-- fat: double (nullable = true)
 |-- sodium: double (nullable = true)
 |-- cakeweek: double (nullable = true)
 |-- wasteless: double (nullable = true)
 |-- 22_minute_meals: double (nullable = true)
 |-- 3_ingredient_recipes: double (nullable = true)
 |-- 30_days_of_groceries: double (nullable = true)
 |-- advance_prep_required: double (nullable = true)
 |-- alabama: double (nullable = true)
 |-- alaska: double (nullable = true)
 |-- alcoholic: double (nullable = true)
 |-- almond: double (nullable = true)
 |-- amaretto: double (nullable = true)
 |-- anchovy: double (nullable = true)
 |-- anise: double (nullable = true)
 |-- anniversary: double (nullable = true)
 |-- anthony_bourdain: double (nullable = true)
 |-- aperitif: double (nullable = true)
 |-- appetizer: double (nullable = true)
 |-- apple: double (nullable = true)
 |

#### 1.2 Explore and create features

Exploring data for machine learning is similar to exploring data when performing a transformation in the sense that we manipulate the data to uncover some inconsistencies, patterns, or gaps

In [10]:
food.show(5)

25/11/09 18:52:19 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------+------+--------+-------+----+------+--------+---------+---------------+--------------------+--------------------+---------------------+-------+------+---------+------+--------+-------+-----+-----------+----------------+--------+---------+-----+-----------+-------+-------+---------+-------+----------+---------+-----+-------+---------+-------+--------------+------------+-----+----+------+------+-----+----+------------+----+----+--------+----------+---------------+----+----+-----------+-----+-------------+--------+-------+-------+----------+-------+-----------+---------+----+--------+-----------+----------+------+-------+------+----+------+-----+-----------+---------+----+-----+-------+--------+-------------+-----+--------+----------+-------+------+--------------+-------+------+--------+------+-------+------+----------+----------------+--------------------+-------+----+----------+--------+---------+-------+-------+------+-----+-----------------+----------+------+----

Identifying your variables as categorical (with the proper subtype) or continuous has a
direct impact on the data preparation and, down the road, the performance of your ML
model. Looking at our summary data, it seems that we have a lot of potentially binary columns.
In the case of the clove column, the minimum and three quartile values are all
zero. To verify this, we’ll group the entire data frame and collect a set of distinct values.
If we have only two values for a given column, binary it is!

In [11]:
def check_for_binary_columns(df):
    """
    Function to generate the necessary PySpark aggregation
    expressions for checking if a column is binary (has exactly 2 distinct values).
    """
    # Step 1: Generate the list of aggregation expressions
    # The string generation here simulates the PySpark expression object syntax.
    binary_check_expressions = [
        (F.size(F.collect_set(F.col(col_name))) == 2).alias(col_name)
        for col_name in food.columns
]
    
    # Step 2: Perform the aggregation on the DataFrame
    is_binary_df = df.agg(*binary_check_expressions)
    
    return is_binary_df

# Execute the check using the refactored function
is_binary_df = check_for_binary_columns(food)

In [12]:
is_binary_df.show()

+-----+------+--------+-------+-----+------+--------+---------+---------------+--------------------+--------------------+---------------------+-------+------+---------+------+--------+-------+-----+-----------+----------------+--------+---------+-----+-----------+-------+-------+---------+-------+----------+---------+-----+-------+---------+-------+--------------+------------+-----+----+------+------+-----+----+------------+----+----+--------+----------+---------------+----+----+-----------+-----+-------------+--------+-------+-------+----------+-------+-----------+---------+----+--------+-----------+----------+------+-------+------+----+------+-----+-----------+---------+----+-----+-------+--------+-------------+-----+--------+----------+-------+------+--------------+-------+------+--------+------+-------+------+----------+----------------+--------------------+-------+----+----------+--------+---------+-------+-------+------+-----+-----------------+----------+------+-------+--------+-

#### 1.3 Data mishapes and feature set

Some columns are not
consistent compared to other related (binary) columns. We are going to explore the content of
the suspicious columns, address the gaps, and continue our exploration. We aim a
more consistent, more robust feature set that will lead to a better ML model.

In [13]:
food.agg(*[F.collect_set(x) for x in ("cakeweek", "wasteless")]).show(1, False)

+-------------------------------+----------------------+
|collect_set(cakeweek)          |collect_set(wasteless)|
+-------------------------------+----------------------+
|[0.0, 880.0, 1.0, 1188.0, 24.0]|[0.0, 1.0, 1439.0]    |
+-------------------------------+----------------------+



In [14]:
food.where("cakeweek > 1.0 or wasteless > 1.0").select("title", "rating", "wasteless", "cakeweek", food.columns[-1]).show()

+--------------------+--------------------+---------+--------+------+
|               title|              rating|wasteless|cakeweek|turkey|
+--------------------+--------------------+---------+--------+------+
|"Beet Ravioli wit...| Aged Balsamic Vi...|      0.0|   880.0|   0.0|
|"Seafood ""Catapl...|            Vermouth|   1439.0|    24.0|   0.0|
|"""Pot Roast"" of...| Aunt Gloria-Style "|      0.0|  1188.0|   0.0|
+--------------------+--------------------+---------+--------+------+



Our data set had a bunch of quotation marks along with some commas that confused PySpark’s parser. Since
we have a small number of records affected, I did not bother with realigning the data
and deleted them outright. I keep the null values as well.

In [15]:
food = food.where(
    (
        #"if cakeweek and wasteless are both either 0.0, 1.0, or null."
        F.col("cakeweek").isin([0.0, 1.0])
        | F.col("cakeweek").isNull()
    )
    & (
        F.col("wasteless").isin([0.0, 1.0])
        | F.col("wasteless").isNull()
    )
)

In [16]:
#we expect 3 less records:
print(food.count(), len(food.columns))

20054 680


Now that we have identified two binary-in-hiding feature columns, we can identify our feature set and our target variable. The target (or label) is the column containing
the value we want to predict. In our case, the column is aptly named `dessert`.

Let's create all-caps variables containing the four main sets of columns I
care about:
- The identifiers, which are the column(s) that contain the information unique to
each record
- The targets, which are the column(s) (most often one) that contain the value we
wish to predict
- The continuous columns, containing continuous features
- The binary columns, containing binary features

In [17]:
IDENTIFIERS = ["title"]

CONTINUOUS_COLUMNS = [
    "rating",
    "calories",
    "protein",
    "fat",
    "sodium",
]

TARGET_COLUMN = ["dessert"]

BINARY_COLUMNS = [
    x
    for x in food.columns
    if x not in CONTINUOUS_COLUMNS
    and x not in TARGET_COLUMN
    and x not in IDENTIFIERS
]

#### 1.4 Find and delete useless records and input binary features

I will removing two types of records:
- Those where all the features are null
- Those where the target is null

Furthermore, we will impute, meaning that we will provide a default value for, our
binary features. Since each of them are 0/1, where zero is False and one is True, we
equate null to False and fill zero as a default value (not ideal, but reasonable)

In [18]:
#FIRST: remove records with only null values

food = food.dropna(
    how="all",
    subset=[x for x in food.columns if x not in IDENTIFIERS],
)

food = food.dropna(subset=TARGET_COLUMN)

print(food.count(), len(food.columns))

20049 680


In [19]:
#SECOND: impute a default value (0.0) to all binary columns

food = food.fillna(0.0, subset=BINARY_COLUMNS)

print(food.where(F.col(BINARY_COLUMNS[0]).isNull()).count())

0


#### 1.5 Cleaning continuous variables (and extreme values)

We are going to:
- cast the variables and delete wrong values
- review the distribution of numerical columns to account for
extreme or unrealistic values.

⚠ The next steps are not a blueprint to be applied regardless
of the situation/dataset.

In [20]:
def is_numeric_expression(column_name: str) -> F.Column:
    """
    Returns a native PySpark expression to check if a string column's content is numeric.
    It removes numeric components (digits, sign, decimal) and checks if the remainder is empty.
    """

    # Check if the column value is NULL or empty after trim
    is_not_empty_or_null = F.col(column_name).isNull() 
    
    # The check: If the original value is not null/empty, and the remaining_chars
    # is null or empty, then it's numeric.
    # A cleaner and simpler way is often just to check for specific non-numeric characters:
    is_valid_numeric = F.col(column_name).rlike(r'^-?(\d*\.)?\d+$')

    return is_not_empty_or_null | is_valid_numeric

food.filter(
    (~is_numeric_expression("rating")) & (~is_numeric_expression("calories"))
).show()

+--------------------+---------+------------+-------+----+------+--------+---------+---------------+--------------------+--------------------+---------------------+-------+------+---------+------+--------+-------+-----+-----------+----------------+--------+---------+-----+-----------+-------+-------+---------+-------+----------+---------+-----+-------+---------+-------+--------------+------------+-----+----+------+------+-----+----+------------+----+----+--------+----------+---------------+----+----+-----------+-----+-------------+--------+-------+-------+----------+-------+-----------+---------+----+--------+-----------+----------+------+-------+------+----+------+-----+-----------+---------+----+-----+-------+--------+-------------+-----+--------+----------+-------+------+--------------+-------+------+--------+------+-------+------+----------+----------------+--------------------+-------+----+----------+--------+---------+-------+-------+------+-----+-----------------+----------+----

We have a single remaining rogue record that we remove in the next CMD before casting the columns as a double. Our continuous feature columns are now all numerical.

In [21]:
filtered_food = food.filter(
        (is_numeric_expression("rating")) & (is_numeric_expression("calories"))
    )

cols_to_cast = ["rating", "calories"]

#convert dataType of each column in list to string
for column in cols_to_cast:
    filtered_food = filtered_food.withColumn(column, F.col(column).cast("double"))

print(filtered_food.count(), len(filtered_food.columns)) #we should lose just one record!

20048 680


We need to use our judgment for the best course of
action to address this data quality issue. I could filter the records once more, but this
time, I’ll cap the values to the 99th percentile, avoiding extreme (and potentially
wrong) values.

In [22]:
#SECOND: Look for extreme values
distribution_df = filtered_food.select(*CONTINUOUS_COLUMNS).summary(
"mean",
"stddev",
"min",
"1%",
"5%",
"50%",
"95%",
"99%",
"max",
)

distribution_df.show()

+-------+------------------+------------------+------------------+------------------+------------------+
|summary|            rating|          calories|           protein|               fat|            sodium|
+-------+------------------+------------------+------------------+------------------+------------------+
|   mean| 3.714460295291301|6324.0634571930705|100.17385283565179| 346.9398083953107| 6226.927244193346|
| stddev|1.3409187660508954| 359079.8369634015| 3840.680997128737|20458.040344124092|333349.56803702697|
|    min|               0.0|               0.0|               0.0|               0.0|               0.0|
|     1%|               0.0|              18.0|               0.0|               0.0|               1.0|
|     5%|               0.0|              62.0|               0.0|               0.0|               5.0|
|    50%|             4.375|             331.0|               8.0|              17.0|             294.0|
|    95%|               5.0|            1316.0|        

To make things easier, we are going to **hardcode** the maximum acceptable values for each column, and
then I apply those maximums iteratively to my food data frame

In [23]:
maximum = {
    "calories": float(distribution_df.select("calories").collect()[-2].calories),
    "protein": float(distribution_df.select("protein").collect()[-2].protein),
    "fat": float(distribution_df.select("fat").collect()[-2].fat),
    "sodium": float(distribution_df.select("sodium").collect()[-2].sodium),
}

for k, v in maximum.items():
    filtered_food = filtered_food.withColumn(k, F.when(F.col(k) > v, v).otherwise(F.col(k)))

#### 1.6 Remove rare binary features

We are going to remove features that are either too rare or too frequent. Binary features with only a few zeroes or ones are not
helpful in classifying a recipe as a dessert: if every recipe (or no recipe) has a certain
feature as true, then that feature does not discriminate properly, meaning that our
model has no use for it.

In last section, we computed the sum of each
binary column; this will give us the numbers of 1.0's since the sum of the ones is equal to their count.

For this model, let's use 10 as threshold.

In [24]:
inst_sum_of_binary_columns = [
    F.sum(F.col(x)).alias(x) for x in BINARY_COLUMNS
]

sum_of_binary_columns = (
    filtered_food.select(*inst_sum_of_binary_columns).head().asDict()  # Since a row is just like a Python dictionary, I can bring the row back to the driver and process it locally.
)

num_rows = filtered_food.count()
too_rare_features = [
    k
    for k, v in sum_of_binary_columns.items()
    if v < 10 or v > (num_rows - 10)
]

print('count of variables to remove: ', len(too_rare_features))

print('\n\n\nvariables to remove: \n',too_rare_features)

#Rather than deleting the columns from the data frame, I just remove them from my BINARY_COLUMNS list.
BINARY_COLUMNS = list(set(BINARY_COLUMNS) - set(too_rare_features))

print('\n\n\n\nvariable kept: \n',BINARY_COLUMNS)

25/11/09 18:53:00 WARN DAGScheduler: Broadcasting large task binary with size 1489.0 KiB


count of variables to remove:  167



variables to remove: 
 ['cakeweek', 'wasteless', '30_days_of_groceries', 'alabama', 'alaska', 'anthony_bourdain', 'apple_juice', 'arizona', 'aspen', 'atlanta', 'australia', 'beverly_hills', 'biscuit', 'boston', 'bran', 'brooklyn', 'brownie', 'buffalo', 'bulgaria', 'burrito', 'cambridge', 'camping', 'canada', 'caviar', 'chicago', 'chili', 'cobbler_crumble', 'columbus', 'cook_like_a_diner', 'cookbook_critic', 'costa_mesa', 'cranberry_sauce', 'crêpe', 'crme_de_cacao', 'cuba', 'cupcake', 'custard', 'dallas', 'denver', 'digestif', 'dominican_republic', 'dorie_greenspan', 'eau_de_vie', 'egg_nog', 'egypt', 'emeril_lagasse', 'england', 'entertaining', 'epi__ushg', 'epi_loves_the_microwave', 'flat_bread', 'frankenrecipe', 'freezer_food', 'friendsgiving', 'frittata', 'fritter', 'germany', 'grains', 'grand_marnier', 'granola', 'grappa', 'guam', 'haiti', 'hamburger', 'hawaii', 'healdsburg', 'hollywood', 'house_cocktail', 'houston', 'hummus', 'iced_coffee', 'id

(We removed 167 features that are either too rare or too frequent.)

## 2. Feature Engineering

Now we are going into two important steps of model building: feature creation (also called
feature engineering) and refinement.

Our goal:
- Creating a few custom features using our continuous feature columns
- Measuring correlation over original and generated continuous features

#### 2.1 Customs features

In PySpark, creating
new features is done simply by creating columns with the information you want;
this means you can create simple or highly sophisticated features.

Just as an example, we’ll take the `protein` and `fat` columns representing
the quantity (in grams) of protein and fat in the recipe, respectively. With the information
in those two columns, I create two features representing the percentage of calories
attributed to each macro nutriment.

⚠ PAY ATENTION TO MULTICOLLINEARITY! When
using a model that has a linear component, such as the linear regression and the
logistic regression, this will cause problems with your model’s accuracy

In [25]:
filtered_food = filtered_food.withColumn(
    "protein_ratio", F.col("protein") * 4 / F.col("calories")  # <1>
).withColumn(
    "fat_ratio", F.col("fat") * 9 / F.col("calories")
) #There are 4 kcal per grams of protein and 9 kcal per grams of fat.

filtered_food = filtered_food.fillna(0.0, subset=["protein_ratio", "fat_ratio"])

CONTINUOUS_COLUMNS += ["protein_ratio", "fat_ratio"]

#### 2.2 Feature correlation

Look at the correlation
between our set of continuous may help us improve our model accuracy and explainability

In this section we are going to address:
1. How PySpark computes the correlation between variables and provides the results in a matrix using Vector and Matrix objects
2. How we can extract values from them. 
3. The correlation between our continuous variables and made a decision about their inclusion in our first model

For computing correlation between variables, PySpark provides the `Correlation` object.
Correlation has a single method, `corr`, that computes the correlation between features
in a `Vector`. Vectors are like PySpark arrays but with a special representation optimized
for ML work.
We are going to use the `VectorAssembler` transformer on the food data frame to create a new column,
continuous_features, that contains a Vector of all our continuous features.
A transformer is a preconfigured object that, as its name indicates, transforms a
data frame. Independently, it looks like unnecessary complexity, but it shines when
applied within a pipeline.

In [26]:
filtered_food.select(CONTINUOUS_COLUMNS).printSchema()

root
 |-- rating: double (nullable = true)
 |-- calories: double (nullable = true)
 |-- protein: double (nullable = true)
 |-- fat: double (nullable = true)
 |-- sodium: double (nullable = true)
 |-- protein_ratio: double (nullable = false)
 |-- fat_ratio: double (nullable = false)



In [27]:
from pyspark.ml.feature import VectorAssembler

continuous_features = VectorAssembler(
    inputCols=CONTINUOUS_COLUMNS, outputCol="continuous_features"
)

vector_food = filtered_food.select(CONTINUOUS_COLUMNS)

for x in CONTINUOUS_COLUMNS:
    vector_food = vector_food.where(~F.isnull(F.col(x))) 

vector_variable = continuous_features.transform(vector_food)

vector_variable.select("continuous_features").show(3, False)

#Correlation will not work well if you blend categorical and/or binary features together.

+---------------------------------------------------------------------+
|continuous_features                                                  |
+---------------------------------------------------------------------+
|[2.5,426.0,30.0,7.0,559.0,0.28169014084507044,0.14788732394366197]   |
|[4.375,403.0,18.0,23.0,1439.0,0.17866004962779156,0.5136476426799007]|
|[3.75,165.0,6.0,7.0,165.0,0.14545454545454545,0.38181818181818183]   |
+---------------------------------------------------------------------+
only showing top 3 rows


Now, we are going to apply the `Correlation.corr()` function on the continuous feature
vector and export the correlation matrix into an easily interpretable pandas Data-
Frame. PySpark returns the correlation matrix in a `DenseMatrix` column type, which
is like a two-dimensional vector. In order to extract the values in an **easy-to-read format**:
1. We extract a single record as a list of Row using head().
2. A Row is like an ordered dictionary, so we can access the first (and only) field
containing our correlation matrix using list slicing.
3. A DenseMatrix can be converted into a pandas-compatible array by using the
toArray() method on the matrix.
4. We can directly create a pandas DataFrame from our Numpy array. Inputting
our column names as an index (in this case, they’ll play the role of “row names”)
makes our correlation matrix very readable.

In [28]:
from pyspark.ml.stat import Correlation

#The corr method takes a data frame and a Vector column reference as a parameter and generates a single-row, single column data frame containing the correlation matrix.
correlation = Correlation.corr(
    vector_variable, "continuous_features"
)

correlation.printSchema()
#DenseMatrix is not easily accessible by itself.

25/11/09 18:53:07 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


root
 |-- pearson(continuous_features): matrix (nullable = false)



In [29]:
import pandas as pd

correlation_array = correlation.head()[0].toArray()

correlation_pd = pd.DataFrame(
    correlation_array,  
    index=CONTINUOUS_COLUMNS,  
    columns=CONTINUOUS_COLUMNS, 
)

print(correlation_pd.iloc[:, :6])

                 rating  calories   protein       fat    sodium  protein_ratio
rating         1.000000  0.102257  0.113292  0.111536  0.065225       0.094429
calories       0.102257  1.000000  0.757837  0.918052  0.516818       0.164735
protein        0.113292  0.757837  1.000000  0.664899  0.585450       0.600182
fat            0.111536  0.918052  0.664899  1.000000  0.421920       0.125572
sodium         0.065225  0.516818  0.585450  0.421920  1.000000       0.339067
protein_ratio  0.094429  0.164735  0.600182  0.125572  0.339067       1.000000
fat_ratio      0.129946  0.176823  0.109188  0.424986  0.033819       0.024854


**There is no absolute threshold for keeping or removing correlated variables.**

We see high correlation between `sodium`,
`calories`, `protein`, and `fat`. Surprisingly, we see little correlation between our custom
features and the columns that contributed to their creation

## 3. Feature Preparation

This section provides an overview of transformers and estimators in the context of feature
preparation. We use transformers and estimators as an abstraction over common
operations in machine learning modeling. We explore two relevant examples of transformers
and estimators:
-  Null imputation, where we provide a value to replace null occurrences in a column
(e.g., the mean)
-  Scaling features, where we normalize the values of a column, so they are on a
more logical scale (e.g., between zero and one)


The best way to think about a `transformer` is by translating its behavior into a
`function`. Below we compare a `VectorAssembler` to a `function` assemble_
vector() that performs the same work, which is to create a Vector named after the
argument to outputCol, which contains all the values in the columns passed to
inputCols. Don’t focus on the actual work here, but more on the mechanism
of application.

![image](files/tables/transformer.jpg)

The transformer object has a two-staged process. 
- First, when instantiating the
transformer, we provide the parameters necessary for its application, but not the data
frame on which it’ll be applied. This echoes the separation of data and instructions we
saw in previously Labs. 
- Then, we use the instantiated transformer’s transform() method on
the data frame to get a transformed data frame.


This separation of instructions and data is key in creating serializable ML pipelines,
which leads to easier ML experiments and model portability

#### 3.1 Imputer estimator

In this section, we cover the Imputer estimator and introduce the concept of an estimator.
Estimators are the main abstraction used by Spark for any data-dependent transformation,
including ML models, so they are pervasive in any ML code using PySpark.

We want our Imputer to impute the mean value to every record in the
calories, protein, fat, and sodium columns when the record is null.

More information in section Imputer: https://spark.apache.org/docs/latest/ml-features

In [30]:
from pyspark.ml.feature import Imputer

OLD_COLS = ["calories", "protein", "fat", "sodium"]
NEW_COLS = ["calories_i", "protein_i", "fat_i", "sodium_i"]

imputer = Imputer(
    strategy="mean",  
    inputCols=OLD_COLS,  
    outputCols=NEW_COLS,  
)

imputer_model = imputer.fit(filtered_food)

CONTINUOUS_COLUMNS = (
    list(set(CONTINUOUS_COLUMNS) - set(OLD_COLS)) + NEW_COLS  
)

In [31]:
#Let's check!

food_imputed = imputer_model.transform(filtered_food)

food_imputed.where("calories is null").select("calories", "calories_i").show(5, False)

+--------+-----------------+
|calories|calories_i       |
+--------+-----------------+
|NULL    |475.5222194325885|
|NULL    |475.5222194325885|
|NULL    |475.5222194325885|
|NULL    |475.5222194325885|
|NULL    |475.5222194325885|
+--------+-----------------+
only showing top 5 rows


#### 3.2 Scaling features

This section covers variable scaling using the MinMaxScaler transformer. Scaling variables
means performing a mathematical transformation on the variables so that they
are all on the same numeric scale.

To choose the right scaling algorithm, we need to look at our variables as a whole.
Since we have so many binary variables, it is convenient to have every variable be
between zero and one. Our protein_ratio and fat_ratio are ratios between zero
and one too!

In [32]:
from pyspark.ml.feature import MinMaxScaler

CONTINUOUS_NB = [x for x in CONTINUOUS_COLUMNS if "ratio" not in x]

continuous_assembler = VectorAssembler(
    inputCols=CONTINUOUS_NB, outputCol="continuous"
)

food_features = continuous_assembler.transform(food_imputed)

continuous_scaler = MinMaxScaler(
    inputCol="continuous",
    outputCol="continuous_scaled",
)

food_features = continuous_scaler.fit(food_features).transform(
    food_features
)

food_features.select("continuous_scaled").show(3, False)


+-----------------------------------------------------------------------------------------+
|continuous_scaled                                                                        |
+-----------------------------------------------------------------------------------------+
|[0.5,0.13300031220730565,0.17341040462427745,0.033816425120772944,0.09874580462815757]   |
|[0.875,0.12581954417733376,0.10404624277456646,0.1111111111111111,0.2541953718424307]    |
|[0.75,0.051514205432407124,0.03468208092485549,0.033816425120772944,0.029146793852676208]|
+-----------------------------------------------------------------------------------------+
only showing top 3 rows


👍 TIP check the pyspark.ml.feature module for other scalers. https://spark.apache.org/docs/2.3.1/api/python/pyspark.ml.html

## 4. Finally, ML Pipeline

We may say an ML pipeline is
an ordered list of transformers and estimators.

#### 4.1 Transformers and estimators


TRANSFORMERS:

Transformer’s sole purpose—through its `transform()` method—is to take
the values in `inputCols` (assembled values) and return a single column, named
`outputCol`, that contains a vector of all the assembled values.
A transformer has a set of explicit parameters (called
Params in the Spark language) that drive its behavior.
Some parameters have a default value in case you
don’t define a value yourself (e.g., handleInvalid).

The most important method of a
transformer is the `transform( )`
method. This method takes a data
frame as an input and returns
a transformed data frame.

Example: `VectorAssembler` is a transformer. Params: inputCols, outputCol, handleInvalid

If you look at the signature for VectorAssembler, you’ll see an asterisk at the beginning
of the parameters list:

`` class pyspark.ml.feature.VectorAssembler(*, inputCols=None,
outputCol=None, handleInvalid='error')`` 

In Python, every parameter after the asterisk (*) is called a keyword-only argument,
meaning that we need to mention the keyword. For instance, we couldn’t do Vector-
Assembler("input_column", "output_column"). For more: https://peps.python.org/pep-3102/

In [33]:
print(continuous_assembler.outputCol)

VectorAssembler_c3312590f738__outputCol


In [34]:
print(continuous_assembler.getOutputCol())

print('\n', continuous_assembler.explainParam("outputCol"))

print('\n', continuous_assembler.explainParams())

continuous

 outputCol: output column name. (default: VectorAssembler_c3312590f738__output, current: continuous)

 handleInvalid: How to handle invalid data (NULL and NaN values). Options are 'skip' (filter out rows with invalid data), 'error' (throw an error), or 'keep' (return relevant number of NaN in the output). Column lengths are taken from the size of ML Attribute Group, which can be set using `VectorSizeHint` in a pipeline before `VectorAssembler`. Column lengths can also be inferred from first rows of the data since it is safe to do so but only in case of 'error' or 'skip'). (default: error)
inputCols: input column names. (current: ['rating', 'calories_i', 'protein_i', 'fat_i', 'sodium_i'])
outputCol: output column name. (default: VectorAssembler_c3312590f738__output, current: continuous)


ESTIMATOR:

Where a transformer transforms an
input data frame into an output data frame, an estimator is fitted on an input data
frame and returns an output transformer.

We focus on estimator usage through the `fit()`
method (versus `transform()` for the transformer), which is really the only notable
difference for the end user. The `fit()` method takes a data
frame as an input and returns a parametrized
transformer as an output.

Just like a transformer, an estimator has a set
of explicit parameters (called Params in the
Spark language) that drive its behavior. Some
parameters have a default value in case you
don’t define a value yourself (e.g., min/max).

Example: `MinMaxScaler` is a estimator. Params: `min`, `max`, `inputcCol`, `outputCol`.

This fit()/transform() approach applies for estimators that are far more complex
than MinMaxScaler. Case in point: ML models are actually implemented as estimators
in Spark.

#### 4.2 Building a complete ML pipeline

This section we will introduce the `Pipeline` object as an estimator with a special purpose:
running other transformers and estimators.

Pipelines build on transformers and estimators
to make training, evaluating, and optimizing ML models much clearer and
more explicit.

ML pipelines are implemented through the Pipeline class, which
is a specialized version of the estimator. The Pipeline estimator has only one
Param, called stages, which takes a list of transformers and estimators.

In [35]:
#Just as a matter of completeness, we are going to repeat the estimators and transformers here, to consolidate the code 

from pyspark.ml import Pipeline
import pyspark.ml.feature as MF

imputer = MF.Imputer(  
    strategy="mean",
    inputCols=["calories", "protein", "fat", "sodium"],
    outputCols=["calories_i", "protein_i", "fat_i", "sodium_i"],
)

continuous_assembler = MF.VectorAssembler(  
    inputCols=["rating", "calories_i", "protein_i", "fat_i", "sodium_i"],
    outputCol="continuous",
)

continuous_scaler = MF.MinMaxScaler(  
    inputCol="continuous",
    outputCol="continuous_scaled",
)

#The food_pipeline pipeline contains three stages, encoded in the stages Param
food_pipeline = Pipeline(  
    stages=[imputer, continuous_assembler, continuous_scaler]
)

In practical terms, since the pipeline is an estimator, it has a `fit()` method that generates
a PipelineModel. Under the hood, the pipeline applies each stage in order, calling
the appropriate method depending on if the stage is a transformer (`transform()`)
or an estimator (`fit()`). By wrapping all of our individual stages into a pipeline, we
only have one method to call, `fit()`, knowing that PySpark will do the right thing to
yield a PipelineModel.

If the stage is
a transformer, it gets applied on the data
and then passed as a stage in the
PipelineModel. If the stage is an estimator,
it gets fitted on the data and the resulting
model gets passed as a stage in the
PipelineModel.

##### 4.2.1 Final dataset (vector column type)

This section will cover the assembly into a final feature vector, the last stage before
sending our data for training.

PySpark requires all the data fed into a machine learning
estimator, as well as some other estimators like the MinMaxScaler, to be in a single vector
column.

REMEMBER: We already know how to assemble data into a vector: use the `VectorAssembler`.

We will assemble all of our BINARY_COLUMNS, the _ratio columns, and the continuous_
scaled vector column from our pipeline. PySpark will do the right thing when assembling
vector columns in another vector: rather than getting nested vectors, the assembly
step will flatten everything into a single, ready-to-use vector.

In [36]:
preml_assembler = MF.VectorAssembler(
    inputCols=BINARY_COLUMNS 
    + ["continuous_scaled"]
    + ["protein_ratio", "fat_ratio"],
    outputCol="features",
)

food_pipeline.setStages(
    [imputer, continuous_assembler, continuous_scaler, preml_assembler]
)

food_pipeline_model = food_pipeline.fit(filtered_food)  # food_pipeline_model becomes a PipelineModel
food_features = food_pipeline_model.transform(filtered_food) 

Our data frame is ready for machine learning! We have a number of records, each with
- A target (or label ) column, dessert, containing a binary input (1.0 if the recipe
is a dessert, 0.0 otherwise)
- A vector of features, called features, containing all the information we want to
train our machine learning model with

In [37]:
food_features.select("title", "dessert", "features").show(5, truncate=30)

+------------------------------+-------+------------------------------+
|                         title|dessert|                      features|
+------------------------------+-------+------------------------------+
|Lentil, Apple, and Turkey W...|    0.0|(513,[11,38,42,66,151,173,3...|
|Boudin Blanc Terrine with R...|    0.0|(513,[2,30,109,110,180,188,...|
| Potato and Fennel Soup Hodge |    0.0|(513,[24,69,150,200,343,436...|
|Mahi-Mahi in Tomato Olive S...|    0.0|(513,[23,42,70,81,102,109,1...|
|     Spinach Noodle Casserole |    0.0|(513,[3,24,52,109,356,361,3...|
+------------------------------+-------+------------------------------+
only showing top 5 rows


We provide 513 distinct features (see the 513 at the beginning of the features column value) with a large number of zeroes. This is
called a sparse features set. When storing vectors, PySpark has two choices for representing
vectors:
-  A dense representation, where a Vector in PySpark is simply a NumPy (a highperformance
multidimensional array library for Python) single-dimensional
array object
-  A sparse representation, where a Vector in PySpark is an optimized sparse vector
compatible with the SciPy (a scientific computing library in Python)
scipy.sparse matrix.

For more: https://www.youtube.com/watch?v=oGwEv82ifrE


PySpark allows for a metadata dictionary to be attached to a
column, let's have a look:

In [38]:
print(food_features.schema["features"])

StructField('features', VectorUDT(), True)


In [39]:
print(food_features.schema["features"].metadata)

# Since they originate from a VectorAssembler, PySpark gives scaled variables a generic name, but you can retrieve their name from the original vector column (here continuous_assembled) as needed.

{'ml_attr': {'num_attrs': 513, 'attrs': {'numeric': [{'name': 'summer', 'idx': 0}, {'name': 'rutabaga', 'idx': 1}, {'name': 'dried_fruit', 'idx': 2}, {'name': 'cheese', 'idx': 3}, {'name': 'lamb', 'idx': 4}, {'name': 'watermelon', 'idx': 5}, {'name': 'no_sugar_added', 'idx': 6}, {'name': 'buttermilk', 'idx': 7}, {'name': 'tofu', 'idx': 8}, {'name': 'bon_apptit', 'idx': 9}, {'name': 'frangelico', 'idx': 10}, {'name': 'sandwich', 'idx': 11}, {'name': 'molasses', 'idx': 12}, {'name': 'bourbon', 'idx': 13}, {'name': 'deep_fry', 'idx': 14}, {'name': 'sandwich_theory', 'idx': 15}, {'name': 'whole_wheat', 'idx': 16}, {'name': 'broccoli_rabe', 'idx': 17}, {'name': 'tailgating', 'idx': 18}, {'name': 'legume', 'idx': 19}, {'name': 'tarragon', 'idx': 20}, {'name': 'passion_fruit', 'idx': 21}, {'name': 'banana', 'idx': 22}, {'name': 'kosher', 'idx': 23}, {'name': 'dairy', 'idx': 24}, {'name': 'flaming_hot_summer', 'idx': 25}, {'name': 'cod', 'idx': 26}, {'name': 'thyme', 'idx': 27}, {'name': 'ice_

##### 4.2.2 Training the model (using a Logistic Regression)

It is time do add a ML model to our Pipeline!

ATTENTION: in real world, you need to choose the correct model to apply to the business problem
you are trying to solve

In our case, because our target is binary (0.0 or 1.0), we restrict ourselves to a classification algorithm. The logistic regression algorithm, despite its name, is a classification algorithm that
belongs to the family of generalized linear models.

Before integrating our logistic regression into our pipeline, we need to create the `estimator`.
This `estimator` is called `LogisticRegression` and comes from the `pyspark.ml
.classification` module. The API documentation page for the LogisticRegression: https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.classification.LogisticRegression.html

In [40]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features", labelCol="dessert", predictionCol="prediction"
)

food_pipeline.setStages(
    [
        imputer,
        continuous_assembler,
        continuous_scaler,
        preml_assembler,
        lr,  # <1>
    ]
)

#We just setted three Params:
# - featuresCol: the column containing our features vector
# - labelCol: the column containing our label (or target)
# - predictionCol: the column that will contain the predictions of our model

Pipeline_d9714130cffb

Below, we `fit()` our pipeline. Before doing so, we need to split our data set into two portions using `randomSplit()`: one for training, which
we feed to our pipeline, and one for testing, which is what we use to evaluate our
model fit.

But before fitting our pipeline, we cache() the training data frame. We do this because ML uses the data frame
repeatedly, so caching in memory provides an increase in speed if your cluster **has
enough memory**.

*Although PySpark will use the same seed, which should guarantee
that the split will be consistent across runs, there are some cases where PySpark
will break that consistency. If you want to be 100% certain about your splits,
split your data frame, write each one to disk, and then read them from the
disk location.*

In [41]:
train, test = filtered_food.randomSplit([0.7, 0.3], 42) 

train.cache()

food_pipeline_model = food_pipeline.fit(train)
results = food_pipeline_model.transform(test)

In [42]:
results.select("prediction", "rawPrediction", "probability").show(3, False)

+----------+----------------------------------------+-------------------------------------------+
|prediction|rawPrediction                           |probability                                |
+----------+----------------------------------------+-------------------------------------------+
|0.0       |[16.04437767474711,-16.04437767474711]  |[0.9999998923496953,1.076503046704147E-7]  |
|0.0       |[16.9224761397766,-16.9224761397766]    |[0.9999999552635033,4.473649672931401E-8]  |
|0.0       |[31.553671602650596,-31.553671602650596]|[0.9999999999999802,1.9761969838327786E-14]|
+----------+----------------------------------------+-------------------------------------------+
only showing top 3 rows


#### 4.3 Evaluate and optimize

In this section, we perform a reviewing of our model results and tuning their implementation.

#####4.3.1 Assessing model accuracy: Confusion matrix and evaluator object

In [43]:
results.groupby("dessert").pivot("prediction").count().show()

#The confusion matrix shows that our data set has a lot more non-desserts than desserts. In the classification world this is called an imbalanced data set

25/11/09 18:54:19 WARN DAGScheduler: Broadcasting large task binary with size 1104.6 KiB
25/11/09 18:54:19 WARN DAGScheduler: Broadcasting large task binary with size 1104.2 KiB
25/11/09 18:54:19 WARN DAGScheduler: Broadcasting large task binary with size 1100.3 KiB
25/11/09 18:54:23 WARN DAGScheduler: Broadcasting large task binary with size 1106.6 KiB
25/11/09 18:54:23 WARN DAGScheduler: Broadcasting large task binary with size 1110.5 KiB


+-------+----+---+
|dessert| 0.0|1.0|
+-------+----+---+
|    0.0|4707| 78|
|    1.0|  86|944|
+-------+----+---+



In Spark 3.1, we now have access to a new `LogisticRegressionSummary` object that avoids the trip to
the RDD world.

We need to first extract our fitted model
from the pipeline model. For this, we can use the stages attribute of `pipeline_
food_model` and access just the last item. From that model, called `lr_model` in the CMD below, we call `evaluate()` on the results data set. `evaluate()` will error out any prediction
columns that exist, so I simply give the relevant ones (dessert, features) to
it. It’s a small price to pay to avoid computing the metrics by hand. Note that PySpark
does not know which label we consider positive and negative. Because of this, the precision
and recall are accessible through `precisionByLabel` and `recallByLabel`,
which both return lists of precision/recall for each label in order.

In [44]:
lr_model = food_pipeline_model.stages[-1]
metrics = lr_model.evaluate(results.select("title", "dessert", "features"))

# LogisticRegressionTrainingSummary

print(f"Model precision: {metrics.precisionByLabel[1]}") 
print(f"Model recall: {metrics.recallByLabel[1]}")

Model precision: 0.923679060665362
Model recall: 0.916504854368932


The receiver operating characteristic curve (ROC) is another common metric used when evaluating binary classification
models. 

The ROC curve is obtained through the BinaryClassificationEvaluator object.
Below we instantiate the said object, asking explicitly for the areaUnderROC metric.

In [45]:
#As homework, you may try build this ROC curve using matplotlib

from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol="dessert",  # <1>
    rawPredictionCol="rawPrediction",  # <1>
    metricName="areaUnderROC",
)

accuracy = evaluator.evaluate(results)
print(f"Area under ROC = {accuracy} ")

Area under ROC = 0.9926806058577062 


##### 4.3.2 Optimizing hyperparameters with cross-validation

By fine-tuning some aspects of the model training (how Spark builds the
fitted model), we can hope to yield better model accuracy. For this, we use a technique
called cross-validation. Cross-validation resamples the data set into training and
testing sets to assess the ability of the model to generalize over new data.

To build the set of hyperparameters we wish to evaluate our model against, we use the
ParamGridBuilder, which assists in creating a Param Map

In [46]:
from pyspark.ml.tuning import ParamGridBuilder

grid_search = (
    ParamGridBuilder() 
    .addGrid(lr.elasticNetParam, [0.0, 1.0]) 
    .build()
)

print(grid_search)

[{Param(parent='LogisticRegression_f9fc8aae29b3', name='elasticNetParam', doc='the ElasticNet mixing parameter, in range [0, 1]. For alpha = 0, the penalty is an L2 penalty. For alpha = 1, it is an L1 penalty.'): 0.0}, {Param(parent='LogisticRegression_f9fc8aae29b3', name='elasticNetParam', doc='the ElasticNet mixing parameter, in range [0, 1]. For alpha = 0, the penalty is an L2 penalty. For alpha = 1, it is an L1 penalty.'): 1.0}]


the output may be a messy, so to facilitate the reading:

 [
 
     {Param(parent='LogisticRegression_14302c005814',
            name='elasticNetParam',
            doc='...'): 0.0},  <4>
     {Param(parent='LogisticRegression_14302c005814',
            name='elasticNetParam',
            doc='...'): 1.0}  <4>

]

Now onto cross-validation. PySpark provides out-of-the-box K-fold crossvalidation
through the CrossValidator class

In [47]:
from pyspark.ml.tuning import CrossValidator

cv = CrossValidator(
    estimator=food_pipeline,
    estimatorParamMaps=grid_search,
    evaluator=evaluator,
    numFolds=3,
    seed=13,
    collectSubModels=True,
)

cv_model = cv.fit(train)

print(cv_model.avgMetrics)

[np.float64(0.9904653624792302), np.float64(0.9904640814583877)]


In [48]:
pipeline_food_model = cv_model.bestModel

In [50]:
# Save pipeline
lr_model.write().overwrite().save("./food_model_pipeline")

#### 4.3 Extracting the coefficientes

This section covers the extraction of our model features and their coefficients. We use
those coefficients to get a sense of the most important features of the model and plan
some improvements for a second iteration.

In [51]:
import pandas as pd

feature_names = ["(Intercept)"] + [ x["name"]
    for x in (
        food_features
        .schema["features"]
        .metadata["ml_attr"]["attrs"]["numeric"]
    )
]

feature_coefficients = [lr_model.intercept] + list(
    lr_model.coefficients.values
)


coefficients = pd.DataFrame(
    feature_coefficients, index=feature_names, columns=["coef"]
)

coefficients["abs_coef"] = coefficients["coef"].abs()

print(coefficients.sort_values(["abs_coef"]))

                     coef   abs_coef
hot_drink        0.000216   0.000216
tequila          0.004069   0.004069
saffron         -0.008822   0.008822
rice            -0.009936   0.009936
frozen_dessert  -0.011764   0.011764
...                   ...        ...
mustard_greens -11.994319  11.994319
sangria        -12.757130  12.757130
rye            -13.232440  13.232440
quinoa         -15.632472  15.632472
arugula        -17.233033  17.233033

[514 rows x 2 columns]


A coefficient close to zero, like kirsch, lemon, and food_processor, means that
this feature is not very predictive of our model. On the flip side, a very high or low
coefficient, like cauliflower, horseradish, and quick_and_healthy, means that this
feature is highly predictive.

# Exercise

We are going to create a model to predict the flight delay over 15 minutes (```ARR_DEL15```) using other attributes - such as, airport code, career, and various weather conditions.

Before starting, you must download the dataset `flight_weather.csv` You will find the dataset on Moodle. Since this is a tabular dataset, you can go to ``Catalog``, then ``tables``. There you can create the table (using the UI option is fine).

> Note : For more accurate learning in classification, use LightGBM classifier in SynapseML library (formerly MMLSpark library).<br>
> Here I use built-in DecisionTree Classifier in MLlib.

*This exercise was based on https://github.com/tsmatz/azure-databricks-exercise*

#### 1.1 Import and clean

In [52]:
#IMPORT DATASET

# File location and type
file_location = "./flight_weather.csv"
file_type = "csv"

# CSV options
infer_schema = "true"
first_row_is_header = "true"
delimiter = ","

# The applied options are for CSV files. For other file types, these will be ignored.
df = spark.read.format(file_type) \
  .option("inferSchema", infer_schema) \
  .option("header", first_row_is_header) \
  .option("sep", delimiter) \
  .load(file_location)

display(df)

DataFrame[_c0: int, X.1: int, YEAR: int, MONTH: int, DAY_OF_MONTH: int, DAY_OF_WEEK: int, FL_DATE: date, UNIQUE_CARRIER: string, TAIL_NUM: string, FL_NUM: int, ORIGIN_AIRPORT_ID: int, ORIGIN: string, ORIGIN_STATE_ABR: string, DEST_AIRPORT_ID: int, DEST: string, DEST_STATE_ABR: string, CRS_DEP_TIME: int, DEP_TIME: double, DEP_DELAY: double, DEP_DELAY_NEW: double, DEP_DEL15: double, DEP_DELAY_GROUP: double, TAXI_OUT: double, WHEELS_OFF: double, WHEELS_ON: double, TAXI_IN: double, CRS_ARR_TIME: int, ARR_TIME: double, ARR_DELAY: double, ARR_DELAY_NEW: double, ARR_DEL15: double, ARR_DELAY_GROUP: double, CANCELLED: int, CANCELLATION_CODE: string, DIVERTED: int, CRS_ELAPSED_TIME: double, ACTUAL_ELAPSED_TIME: double, AIR_TIME: double, FLIGHTS: int, DISTANCE: int, DISTANCE_GROUP: int, CARRIER_DELAY: double, WEATHER_DELAY: double, NAS_DELAY: double, SECURITY_DELAY: double, LATE_AIRCRAFT_DELAY: double, X: string, VisibilityOrigin: double, DryBulbCelsiusOrigin: double, DewPointCelsiusOrigin: doubl

In this dataset,

`ARR_DEL15` : 1 when the flight is delayed over 15 minutes, 0 otherwise.

`XXXOrigin` : Weather conditions in departure airport.

`XXXDest` : Weather conditions in destination airport.

In [53]:
print(df.count(), len(df.columns))

100000 59


In [54]:
df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- X.1: integer (nullable = true)
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY_OF_MONTH: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- FL_DATE: date (nullable = true)
 |-- UNIQUE_CARRIER: string (nullable = true)
 |-- TAIL_NUM: string (nullable = true)
 |-- FL_NUM: integer (nullable = true)
 |-- ORIGIN_AIRPORT_ID: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_STATE_ABR: string (nullable = true)
 |-- DEST_AIRPORT_ID: integer (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_STATE_ABR: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_TIME: double (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- DEP_DELAY_NEW: double (nullable = true)
 |-- DEP_DEL15: double (nullable = true)
 |-- DEP_DELAY_GROUP: double (nullable = true)
 |-- TAXI_OUT: double (nullable = true)
 |-- WHEELS_OFF: double (nullabl

#### 1.2 Explore the features

You may like to explore the feature by using the graphics in the table above

#### 1.3 Data mishapes and feature set

Mark as "delayed over 15 minutes" if it's canceled.

Remove flights if it's diverted.

#### 1.4 Find and delete useless records and input binary features

Narrow to required columns.

Drop rows which has null value in columns.

#### 1.5 Cleaning continuous variables (and extreme values)

Look for extreme values

####4.2.1 Final dataset (vector column type)

Convert categorical values to index values (0, 1, ...) for the following columns.

- Carrier code (```UNIQUE_CARRIER```)
- Airport code in departure (```ORIGIN```)
- Airport code in destination (```DEST```)
- Flag (0 or 1) for delay over 15 minutes (```ARR_DEL15```)

#### 4.2.2 Training the model

####4.3.1 Assessing model accuracy